In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt
import glob
from scipy.stats import t as t_dist, norm

# ------------------------------------------------
# Global plotting style for paper figures
# ------------------------------------------------
plt.rcParams.update({
    # Resolution
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # Fonts (readable when scaled to ~45%)
    "font.size": 12,          # base
    "axes.labelsize": 14,     # axis labels
    "axes.titlesize": 14,     # titles
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # Line & marker thickness
    "axes.linewidth": 1.4,    # axis spines
    "lines.linewidth": 2.0,   # default line width
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,

    # Legend text
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
})

def style_axes(ax):
    """Minimalist, journal-style axes: no grid, bold ticks, thicker spines."""
    ax.grid(False)  # no gridlines

    # Remove top/right spines for a clean look
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Slightly thicker remaining spines
    for spine in ax.spines.values():
        spine.set_linewidth(1.4)

    # Bold tick labels for legibility in small panels
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")



# ------------------------------------------------
# Main
# ------------------------------------------------
def main():
    with open(os.path.join(json_path, "statistics.json"), "r") as f:
        stats_data = json.load(f)

    # Convert "layer_0" -> 0, etc.
    all_results = {}
    for k, v in stats_data.items():
        if isinstance(k, str) and k.startswith("layer_"):
            layer_idx = int(k.split("_")[-1])
        else:
            layer_idx = int(k)
        all_results[layer_idx] = {"statistics": v}

    layers = sorted(all_results.keys())

    # Output directory: ./results_summary/VECTOR_NAME/
    save_root = os.path.join(output_dir, VECTOR_NAME)
    os.makedirs(save_root, exist_ok=True)

    # ------------------------------------------------
    # Helpers
    # ------------------------------------------------
    def to_title_label(text: str) -> str:
        return text.replace("_", " ").title()

    def compute_mean_diff_ci(mean1, sd1, n1, mean2, sd2, n2):
        """Return (mean_diff, lower95, upper95) using Welch SE."""
        md = mean1 - mean2
        se = sqrt(sd1**2 / n1 + sd2**2 / n2)
        ci_low = md - 1.96 * se
        ci_high = md + 1.96 * se
        return md, ci_low, ci_high, se

    def compute_welch_p(mean1, sd1, n1, mean2, sd2, n2):
        """Welch t-test p-value using summary stats."""
        s1_sq = sd1**2
        s2_sq = sd2**2
        se_sq = s1_sq / n1 + s2_sq / n2
        if se_sq == 0:
            return 0.0, np.inf, 0.0  # degenerate
        se = sqrt(se_sq)
        t_val = (mean1 - mean2) / se
        # Welch-Satterthwaite df
        df_num = se_sq**2
        df_den = (s1_sq**2 / (n1**2 * (n1 - 1))) + (s2_sq**2 / (n2**2 * (n2 - 1)))
        df = df_num / df_den if df_den != 0 else (n1 + n2 - 2)
        p_val = 2 * t_dist.sf(abs(t_val), df)
        return t_val, df, p_val

    def cohens_d_ci(d_val, n1, n2):
        """Approximate CI for Cohen's d (independent groups)."""
        N = n1 + n2
        if N <= 2:
            return d_val, d_val, d_val
        se_d = sqrt((N / (n1 * n2)) + (d_val**2 / (2 * (N - 2))))
        ci_low = d_val - 1.96 * se_d
        ci_high = d_val + 1.96 * se_d
        return d_val, ci_low, ci_high

    def auc_ci(auc_val, n1, n2):
        """CI and p-value for AUC vs 0.5 using Hanley & McNeil approx."""
        # Guard for edge cases
        if auc_val <= 0 or auc_val >= 1 or n1 <= 0 or n2 <= 0:
            return auc_val, auc_val, auc_val, 0.0
        Q1 = auc_val / (2 - auc_val)
        Q2 = 2 * auc_val**2 / (1 + auc_val)
        var_auc = (
            auc_val * (1 - auc_val)
            + (n1 - 1) * (Q1 - auc_val**2)
            + (n2 - 1) * (Q2 - auc_val**2)
        ) / (n1 * n2)
        if var_auc <= 0:
            return auc_val, auc_val, auc_val, 0.0
        se = sqrt(var_auc)
        ci_low = auc_val - 1.96 * se
        ci_high = auc_val + 1.96 * se
        z = (auc_val - 0.5) / se
        p_val = 2 * norm.sf(abs(z))
        return auc_val, ci_low, ci_high, p_val

    # ------------------------------------------------
    # Summary Figure (4 panels, with colors for direction/significance)
    # ------------------------------------------------
    def visualize_all_layers_summary(all_results, save_path, group1_name, group2_name):
        layers = sorted(all_results.keys())
        W_ds = [all_results[l]['statistics']['wass']['D'] for l in layers]
        pvals = [all_results[l]['statistics']['wass']['p'] for l in layers]
        g1_means = [all_results[l]['statistics']['group1']['mean'] for l in layers]
        g2_means = [all_results[l]['statistics']['group2']['mean'] for l in layers]
        mean_diffs = [all_results[l]['statistics']['separation']['mean_difference'] for l in layers]
        correct_dirs = [all_results[l]['statistics']['direction']['correct_direction'] for l in layers]

        fig, axes = plt.subplots(2, 2, figsize=(10, 7))  # slightly more compact for panels

        # 1) Wasserstein D across layers (green = correct, red = wrong)
        ax1 = axes[0, 0]
        colors = ['#2E7D32' if cd else '#D32F2F' for cd in correct_dirs]
        for layer, wd, col in zip(layers, W_ds, colors):
            ax1.plot(
                layer,
                wd,
                "o",
                markersize=7,
                color=col,
                markeredgecolor="black",
                markeredgewidth=1.0,
                linewidth=0.0,
            )
        ax1.plot(layers, W_ds, '-', color='gray', alpha=0.4, linewidth=1.5)
        ax1.set_xlabel("Layer", fontsize=14, fontweight="bold")
        ax1.set_ylabel("Wasserstein D", fontsize=14, fontweight="bold")
        ax1.set_title("Wasserstein Distance Across Layers", fontsize=14, fontweight="bold")

        # 2) Mean projections
        ax2 = axes[0, 1]
        ax2.plot(
            layers,
            g1_means,
            "o-",
            color="#336699",
            markersize=6,
            linewidth=2.0,
            label=group1_name,
        )
        ax2.plot(
            layers,
            g2_means,
            "s--",
            color="#CC6633",
            markersize=6,
            linewidth=2.0,
            label=group2_name,
        )
        ax2.axhline(0, linestyle=":", color="gray", linewidth=1.2)
        ax2.set_xlabel("Layer", fontsize=14, fontweight="bold")
        ax2.set_ylabel("Mean Projection", fontsize=14, fontweight="bold")
        ax2.set_title("Means Across Layers", fontsize=14, fontweight="bold")
        legend2 = ax2.legend(frameon=False, loc="best")
        for text in legend2.get_texts():
            text.set_fontweight("bold")

        # 3) Mean difference, colored by direction correctness
        ax3 = axes[1, 0]
        bar_colors = ['#2E7D32' if cd else '#D32F2F' for cd in correct_dirs]
        ax3.bar(
            layers,
            mean_diffs,
            color=bar_colors,
            edgecolor="black",
            linewidth=1.0,
            alpha=0.85,
        )
        ax3.axhline(0, linestyle=":", color="gray", linewidth=1.2)
        ax3.set_xlabel("Layer", fontsize=14, fontweight="bold")
        ax3.set_ylabel(f"Mean Difference ({group1_name} - {group2_name})",
                    fontsize=14, fontweight="bold")
        ax3.set_title("Mean Difference Across Layers", fontsize=14, fontweight="bold")

        # 4) -log10(p)
        ax4 = axes[1, 1]
        min_p = 1e-10
        logp = [-np.log10(max(p, min_p)) for p in pvals]

        colors_p = []
        for lp, cd in zip(logp, correct_dirs):
            if not cd:
                colors_p.append("#D32F2F")   # wrong direction
            elif lp >= 1.3:                  # p <= 0.05
                colors_p.append("#2E7D32")
            elif lp >= 1.0:                  # weaker
                colors_p.append("#FFA000")
            else:
                colors_p.append("#BDBDBD")

        ax4.bar(
            layers,
            logp,
            edgecolor="black",
            color=colors_p,
            linewidth=1.0,
            alpha=0.85,
        )
        ax4.set_xlabel("Layer", fontsize=14, fontweight="bold")
        ax4.set_ylabel(r"$-\log_{10}(p)$", fontsize=14, fontweight="bold")
        ax4.set_title("Significance Across Layers", fontsize=14, fontweight="bold")

        # Apply minimalist styling (no grid, bold ticks, thicker spines)
        for ax in axes.ravel():
            style_axes(ax)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()


    # ------------------------------------------------
    # Minimal standalone metric plots (Cohen's d, WD, AUC)
    # ------------------------------------------------
    def plot_metric(metric_name, values, save_path, add_ci=None):
        plt.figure(figsize=(6, 4))
        plt.plot(
            layers,
            values,
            "o-",
            linewidth=2.0,
            color="black",
            markersize=6,
        )
        if add_ci is not None:
            lower, upper = add_ci
            plt.fill_between(
                layers,
                lower,
                upper,
                alpha=0.15,
                color="gray",
                linewidth=0,
            )

        plt.xlabel("Layer", fontsize=14, fontweight="bold")
        plt.ylabel(metric_name, fontsize=14, fontweight="bold")

        ax = plt.gca()
        style_axes(ax)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()


    # ------------------------------------------------
    # Group mean activations plot (colored for comparison)
    # ------------------------------------------------
    def plot_group_means(layers, g1_means, g2_means, save_path, group1_name, group2_name):
        plt.figure(figsize=(6, 4))
        plt.plot(
            layers,
            g1_means,
            "o-",
            linewidth=2.0,
            color="#1f77b4",
            markersize=6,
            label=to_title_label(group1_name),
        )
        plt.plot(
            layers,
            g2_means,
            "s--",
            linewidth=2.0,
            color="#ff7f0e",
            markersize=6,
            label=to_title_label(group2_name),
        )

        plt.xlabel("Layer", fontsize=14, fontweight="bold")
        plt.ylabel("Mean Projection", fontsize=14, fontweight="bold")

        ax = plt.gca()
        style_axes(ax)

        legend = plt.legend(frameon=False, loc="best")
        for text in legend.get_texts():
            text.set_fontweight("bold")

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()


    # ------------------------------------------------
    # Compute metrics over layers
    # ------------------------------------------------
    cohens_d = []
    wasserstein = []
    auc_vals = []
    mean_diff_ci_low = []
    mean_diff_ci_high = []
    mean_diffs = []
    g1_means_all = []
    g2_means_all = []

    t_vals = []
    welch_dfs = []
    welch_pvals = []

    for l in layers:
        d = all_results[l]["statistics"]
        g1 = d["group1"]
        g2 = d["group2"]

        md, lo, hi, _ = compute_mean_diff_ci(
            g1["mean"], g1["std"], g1["n"],
            g2["mean"], g2["std"], g2["n"]
        )

        mean_diffs.append(md)
        mean_diff_ci_low.append(lo)
        mean_diff_ci_high.append(hi)

        cohens_d.append(d["separation"]["cohen_d"])
        wasserstein.append(d["wass"]["D"])
        auc_vals.append(d["auc"])
        g1_means_all.append(g1["mean"])
        g2_means_all.append(g2["mean"])

        # Welch t-test from summary stats
        t_val, df, p_val = compute_welch_p(
            g1["mean"], g1["std"], g1["n"],
            g2["mean"], g2["std"], g2["n"]
        )
        t_vals.append(t_val)
        welch_dfs.append(df)
        welch_pvals.append(p_val)

    # ------------------------------------------------
    # Use parsed group names
    # ------------------------------------------------
    group1_name = GROUP1_LABEL
    group2_name = GROUP2_LABEL

    # Summary plot
    summary_path = os.path.join(save_root, f"{VECTOR_NAME}_summary.png")
    visualize_all_layers_summary(
        all_results,
        summary_path,
        group1_name=group1_name,
        group2_name=group2_name
    )

    # Metric PDFs (minimal)
    plot_metric("Cohen's d", cohens_d,
                os.path.join(save_root, "cohens_d_layers.pdf"))

    plot_metric("Wasserstein D", wasserstein,
                os.path.join(save_root, "wasserstein_layers.pdf"))

    plot_metric("AUC", auc_vals,
                os.path.join(save_root, "auc_layers.pdf"))

    # "Mean difference" PDF now shows group activations (colored)
    plot_group_means(
        layers,
        g1_means_all,
        g2_means_all,
        os.path.join(save_root, "mean_difference_layers.pdf"),
        group1_name=group1_name,
        group2_name=group2_name
    )

    # ------------------------------------------------
    # Identify best WD layer and build report
    # ------------------------------------------------
    best_layer = layers[int(np.argmax(wasserstein))]
    best_stats = all_results[best_layer]["statistics"]
    g1b = best_stats["group1"]
    g2b = best_stats["group2"]

    # Mean diff, SE, CI and Welch p
    md_best, md_lo, md_hi, se_md = compute_mean_diff_ci(
        g1b["mean"], g1b["std"], g1b["n"],
        g2b["mean"], g2b["std"], g2b["n"]
    )
    t_best, df_best, p_welch = compute_welch_p(
        g1b["mean"], g1b["std"], g1b["n"],
        g2b["mean"], g2b["std"], g2b["n"]
    )

    # WD: value and p from statistics, CI from mean-diff CI
    wd_val = best_stats["wass"]["D"]
    wd_p = best_stats["wass"]["p"]
    wd_ci_low, wd_ci_high = md_lo, md_hi

    # Cohen's d: value from statistics, p from Welch t-test, CI via SE(d)
    d_val = best_stats["separation"]["cohen_d"]
    _, d_ci_low, d_ci_high = cohens_d_ci(d_val, g1b["n"], g2b["n"])
    d_p = p_welch

    # AUC: from statistics + Hanley & McNeil CI/p
    auc_val = best_stats["auc"]
    auc_val, auc_ci_low, auc_ci_high, auc_p = auc_ci(auc_val, g1b["n"], g2b["n"])

    t_best, df_best, p_welch = compute_welch_p(
    g1b["mean"], g1b["std"], g1b["n"],
    g2b["mean"], g2b["std"], g2b["n"]
    )

    report = {
        "layer": int(best_layer),
        "WD": {
            "value": round(float(wd_val), 2),
            "pvalue": round(float(wd_p), 3),
            "95% CI": [round(float(wd_ci_low), 2), round(float(wd_ci_high), 2)],
        },
        "means": {
            "g1": {
                "M": round(g1b["mean"], 2), 
                "SD": round(g1b["std"], 2), 
                "N": round(g1b["n"], 2)
            },
            "g2": {
                "M": round(g2b["mean"], 2), 
                "SD": round(g2b["std"], 2), 
                "N": round(g2b["n"], 2)
            }
        },
        "t_test": {
        "t": round(float(t_best), 2),
        "df": round(float(df_best), 1),
        "pvalue": round(float(p_welch), 3),
        },
        "cohens_d": {
            "value": round(float(d_val), 2),
            "pvalue": round(float(d_p), 3),
            "95% CI": [round(float(d_ci_low), 2), round(float(d_ci_high), 2)],
        },
        "AUC": {
            "value": round(float(auc_val), 2),
            "pvalue": round(float(auc_p), 3),
            "95% CI": [round(float(auc_ci_low), 2), round(float(auc_ci_high), 2)],
        },
    }

    report_path = os.path.join(save_root, "report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    # Print report instead of "Best WD Layer: ..."
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    # ==============================
    # User-defined variables
    # ==============================
    item_names = {
        "realistic_vs_no": "realistic_items_vs_no_realistic_items*",
        "symbolic_vs_no": "symbolic_items_vs_no_symbolic_items*",
        "symbolic_vs_realistic": "symbolic_items_vs_realistic_items*",
        "symbolic_vs_unrelated": "symbolic_items_vs_unrelated_items*",
        "realistic_vs_unrelated": "realistic_items_vs_unrelated_items*"
    }

    VECTOR_NAME = "realistic_vs_unrelated"

    # Pattern used to find the folder and to infer group labels
    pattern_spec = item_names[VECTOR_NAME]  # e.g. "realistic_items_vs_no_realistic_items*"
    pattern = f"./projections/threat_concepts/{VECTOR_NAME}/{pattern_spec}"
    matches = glob.glob(pattern)

    if not matches:
        raise FileNotFoundError(f"No folder matches pattern: {pattern}")

    json_path = matches[0]
    output_dir = "./projection_results_summary/threat_concepts"

    # Derive group labels from pattern_spec
    base = pattern_spec.rstrip("*")  # e.g. "realistic_items_vs_no_realistic_items"
    left, right = base.split("_items_vs_")  # "realistic", "no_realistic_items"
    right = right.rsplit("_items", 1)[0]    # "no_realistic"

    GROUP1_LABEL = left           # e.g. "realistic"
    GROUP2_LABEL = right          # e.g. "no_realistic"

    main()


{
  "layer": 31,
  "WD": {
    "value": 1.08,
    "pvalue": 0.0,
    "95% CI": [
      1.01,
      1.15
    ]
  },
  "means": {
    "g1": {
      "M": 2.01,
      "SD": 0.37,
      "N": 295
    },
    "g2": {
      "M": 0.93,
      "SD": 0.48,
      "N": 299
    }
  },
  "t_test": {
    "t": 30.82,
    "df": 557.1,
    "pvalue": 0.0
  },
  "cohens_d": {
    "value": 2.53,
    "pvalue": 0.0,
    "95% CI": [
      2.31,
      2.74
    ]
  },
  "AUC": {
    "value": 0.97,
    "pvalue": 0.0,
    "95% CI": [
      0.95,
      0.98
    ]
  }
}


Treatments

In [25]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt
from scipy.stats import t as t_dist, norm

# ------------------------------------------------
# Global plotting style for paper figures
# ------------------------------------------------
plt.rcParams.update({
    # Resolution
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # Fonts (readable when scaled to ~45%)
    "font.size": 12,          # base
    "axes.labelsize": 14,     # axis labels
    "axes.titlesize": 14,     # titles
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # Line & marker thickness
    "axes.linewidth": 1.4,    # axis spines
    "lines.linewidth": 2.0,   # default line width
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,

    # Legend text
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
})

def style_axes(ax):
    """Minimalist, journal-style axes: no grid, bold ticks, thicker spines."""
    ax.grid(False)  # no gridlines

    # Remove top/right spines for a clean look
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Slightly thicker remaining spines
    for spine in ax.spines.values():
        spine.set_linewidth(1.4)

    # Bold tick labels for legibility when downscaled
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")



# ------------------------------------------------
# Helpers independent of folder layout
# ------------------------------------------------
def to_title_label(text: str) -> str:
    return text.replace("_", " ").title()


def compute_mean_diff_ci(mean1, sd1, n1, mean2, sd2, n2):
    """Return (mean_diff, lower95, upper95, se) using Welch SE."""
    md = mean1 - mean2
    se = sqrt(sd1**2 / n1 + sd2**2 / n2)
    ci_low = md - 1.96 * se
    ci_high = md + 1.96 * se
    return md, ci_low, ci_high, se


def compute_welch_p(mean1, sd1, n1, mean2, sd2, n2):
    """Welch t-test p-value using summary stats."""
    s1_sq = sd1**2
    s2_sq = sd2**2
    se_sq = s1_sq / n1 + s2_sq / n2
    if se_sq == 0:
        return 0.0, np.inf, 0.0  # degenerate
    se = sqrt(se_sq)
    t_val = (mean1 - mean2) / se
    df_num = se_sq**2
    df_den = (s1_sq**2 / (n1**2 * (n1 - 1))) + (s2_sq**2 / (n2**2 * (n2 - 1)))
    df = df_num / df_den if df_den != 0 else (n1 + n2 - 2)
    p_val = 2 * t_dist.sf(abs(t_val), df)
    return t_val, df, p_val


def cohens_d_ci(d_val, n1, n2):
    """Approximate CI for Cohen's d (independent groups)."""
    N = n1 + n2
    if N <= 2:
        return d_val, d_val, d_val
    se_d = sqrt((N / (n1 * n2)) + (d_val**2 / (2 * (N - 2))))
    ci_low = d_val - 1.96 * se_d
    ci_high = d_val + 1.96 * se_d
    return d_val, ci_low, ci_high


def auc_ci(auc_val, n1, n2):
    """CI and p-value for AUC vs 0.5 using Hanley & McNeil approx."""
    if auc_val <= 0 or auc_val >= 1 or n1 <= 0 or n2 <= 0:
        return auc_val, auc_val, auc_val, 0.0
    Q1 = auc_val / (2 - auc_val)
    Q2 = 2 * auc_val**2 / (1 + auc_val)
    var_auc = (
        auc_val * (1 - auc_val)
        + (n1 - 1) * (Q1 - auc_val**2)
        + (n2 - 1) * (Q2 - auc_val**2)
    ) / (n1 * n2)
    if var_auc <= 0:
        return auc_val, auc_val, auc_val, 0.0
    se = sqrt(var_auc)
    ci_low = auc_val - 1.96 * se
    ci_high = auc_val + 1.96 * se
    z = (auc_val - 0.5) / se
    p_val = 2 * norm.sf(abs(z))
    return auc_val, ci_low, ci_high, p_val


def visualize_all_layers_summary(all_results, save_path, group1_name, group2_name):
    layers = sorted(all_results.keys())
    W_ds = [all_results[l]["statistics"]["wass"]["D"] for l in layers]
    pvals = [all_results[l]["statistics"]["wass"]["p"] for l in layers]
    g1_means = [all_results[l]["statistics"]["group1"]["mean"] for l in layers]
    g2_means = [all_results[l]["statistics"]["group2"]["mean"] for l in layers]
    mean_diffs = [all_results[l]["statistics"]["separation"]["mean_difference"] for l in layers]
    correct_dirs = [all_results[l]["statistics"]["direction"]["correct_direction"] for l in layers]

    # Slightly more compact but still panel-friendly
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))

    # 1) Wasserstein D across layers
    ax1 = axes[0, 0]
    colors = ["#2E7D32" if cd else "#D32F2F" for cd in correct_dirs]
    for layer, wd, col in zip(layers, W_ds, colors):
        ax1.plot(
            layer,
            wd,
            "o",
            markersize=7,
            color=col,
            markeredgecolor="black",
            markeredgewidth=1.0,
            linewidth=0.0,
        )
    ax1.plot(layers, W_ds, "-", color="gray", alpha=0.4, linewidth=1.5)
    ax1.set_xlabel("Layer", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Wasserstein D", fontsize=14, fontweight="bold")
    ax1.set_title("Wasserstein Distance Across Layers", fontsize=14, fontweight="bold")

    # 2) Mean projections
    ax2 = axes[0, 1]
    ax2.plot(
        layers,
        g1_means,
        "o-",
        color="#336699",
        markersize=6,
        linewidth=2.0,
        label=group1_name,
    )
    ax2.plot(
        layers,
        g2_means,
        "s--",
        color="#CC6633",
        markersize=6,
        linewidth=2.0,
        label=group2_name,
    )
    ax2.axhline(0, linestyle=":", color="gray", linewidth=1.2)
    ax2.set_xlabel("Layer", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Mean Projection", fontsize=14, fontweight="bold")
    ax2.set_title("Means Across Layers", fontsize=14, fontweight="bold")
    legend2 = ax2.legend(frameon=False, loc="best")
    for text in legend2.get_texts():
        text.set_fontweight("bold")

    # 3) Mean difference
    ax3 = axes[1, 0]
    bar_colors = ["#2E7D32" if cd else "#D32F2F" for cd in correct_dirs]
    ax3.bar(
        layers,
        mean_diffs,
        color=bar_colors,
        edgecolor="black",
        linewidth=1.0,
        alpha=0.85,
    )
    ax3.axhline(0, linestyle=":", color="gray", linewidth=1.2)
    ax3.set_xlabel("Layer", fontsize=14, fontweight="bold")
    ax3.set_ylabel(f"Mean Difference ({group1_name} - {group2_name})",
                   fontsize=14, fontweight="bold")
    ax3.set_title("Mean Difference Across Layers", fontsize=14, fontweight="bold")

    # 4) -log10(p)
    ax4 = axes[1, 1]
    min_p = 1e-10
    logp = [-np.log10(max(p, min_p)) for p in pvals]

    colors_p = []
    for lp, cd in zip(logp, correct_dirs):
        if not cd:
            colors_p.append("#D32F2F")   # wrong direction
        elif lp >= 1.3:                  # p <= 0.05
            colors_p.append("#2E7D32")
        elif lp >= 1.0:                  # weaker
            colors_p.append("#FFA000")
        else:
            colors_p.append("#BDBDBD")

    ax4.bar(
        layers,
        logp,
        edgecolor="black",
        color=colors_p,
        linewidth=1.0,
        alpha=0.85,
    )
    ax4.set_xlabel("Layer", fontsize=14, fontweight="bold")
    ax4.set_ylabel(r"$-\log_{10}(p)$", fontsize=14, fontweight="bold")
    ax4.set_title("Significance Across Layers", fontsize=14, fontweight="bold")

    # Apply minimalist styling
    for ax in axes.ravel():
        style_axes(ax)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()



def plot_metric(layers, metric_name, values, save_path, add_ci=None):
    plt.figure(figsize=(6, 4))
    plt.plot(
        layers,
        values,
        "o-",
        linewidth=2.0,
        color="black",
        markersize=6,
    )
    if add_ci is not None:
        lower, upper = add_ci
        plt.fill_between(
            layers,
            lower,
            upper,
            alpha=0.15,
            color="gray",
            linewidth=0,
        )

    plt.xlabel("Layer", fontsize=14, fontweight="bold")
    plt.ylabel(metric_name, fontsize=14, fontweight="bold")

    ax = plt.gca()
    style_axes(ax)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_group_means(layers, g1_means, g2_means, save_path, group1_name, group2_name):
    plt.figure(figsize=(6, 4))
    plt.plot(
        layers,
        g1_means,
        "o-",
        linewidth=2.0,
        color="#1f77b4",
        markersize=6,
        label=to_title_label(group1_name),
    )
    plt.plot(
        layers,
        g2_means,
        "s--",
        linewidth=2.0,
        color="#ff7f0e",
        markersize=6,
        label=to_title_label(group2_name),
    )

    plt.xlabel("Layer", fontsize=14, fontweight="bold")
    plt.ylabel("Mean Projection", fontsize=14, fontweight="bold")

    ax = plt.gca()
    style_axes(ax)

    legend = plt.legend(frameon=False, loc="best")
    for text in legend.get_texts():
        text.set_fontweight("bold")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


# ------------------------------------------------
# Folder-name parsing
# ------------------------------------------------
def strip_timestamp(name: str) -> str:
    """
    Optionally strip trailing '_YYYYMMDD_HHMMSS' for robustness.
    If your folders don't have timestamps, this is a no-op.
    """
    parts = name.split("_")
    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():
        return "_".join(parts[:-2])
    return name


def parse_condition_left_right(condition_name: str):
    """
    Given 'both_threat_vs_no_threat', return ('both_threat', 'no_threat').
    """
    if "_vs_" not in condition_name:
        raise ValueError(f"Condition name has no '_vs_': {condition_name}")
    left, right = condition_name.split("_vs_", 1)
    return left, right


def condition_to_output_folder(condition_name: str) -> str:
    """
    Map condition folder name to output folder name.

    Example mapping:
      'both_threat_vs_no_threat' -> 'both_vs_no_threat'
    Everything else stays unchanged.
    """
    toks = condition_name.split("_")
    # both_threat_vs_no_threat -> ["both", "threat", "vs", "no", "threat"]
    if len(toks) >= 5 and toks[1] == "threat" and toks[2] == "vs" and toks[-1] == "threat":
        # pattern: <X>_threat_vs_<Y>_threat -> <X>_vs_<Y>_threat
        out_tokens = [toks[0], "vs"] + toks[3:]
        return "_".join(out_tokens)
    return condition_name


# ------------------------------------------------
# Core processing for ONE condition folder
# ------------------------------------------------
def run_one_projection(json_path, save_root, group1_name, group2_name, vector_name, condition_name):
    stats_file = os.path.join(json_path, "statistics.json")
    if not os.path.isfile(stats_file):
        raise FileNotFoundError(f"statistics.json not found in {json_path}")

    with open(stats_file, "r") as f:
        stats_data = json.load(f)

    # Convert "layer_0" -> 0, etc.
    all_results = {}
    for k, v in stats_data.items():
        if isinstance(k, str) and k.startswith("layer_"):
            layer_idx = int(k.split("_")[-1])
        else:
            layer_idx = int(k)
        all_results[layer_idx] = {"statistics": v}

    layers = sorted(all_results.keys())
    os.makedirs(save_root, exist_ok=True)

    # Collect metrics
    cohens_d_vals = []
    wasserstein_vals = []
    auc_vals = []
    g1_means_all = []
    g2_means_all = []

    for l in layers:
        d = all_results[l]["statistics"]
        g1 = d["group1"]
        g2 = d["group2"]

        cohens_d_vals.append(d["separation"]["cohen_d"])
        wasserstein_vals.append(d["wass"]["D"])
        auc_vals.append(d["auc"])
        g1_means_all.append(g1["mean"])
        g2_means_all.append(g2["mean"])

    # Summary plot
    summary_path = os.path.join(save_root, f"{vector_name}_{condition_name}_summary.png")
    visualize_all_layers_summary(
        all_results,
        summary_path,
        group1_name=group1_name,
        group2_name=group2_name,
    )

    # Metric PDFs
    plot_metric(
        layers,
        "Cohen's d",
        cohens_d_vals,
        os.path.join(save_root, "cohens_d_layers.pdf"),
    )
    plot_metric(
        layers,
        "Wasserstein D",
        wasserstein_vals,
        os.path.join(save_root, "wasserstein_layers.pdf"),
    )
    plot_metric(
        layers,
        "AUC",
        auc_vals,
        os.path.join(save_root, "auc_layers.pdf"),
    )

    # Group means
    plot_group_means(
        layers,
        g1_means_all,
        g2_means_all,
        os.path.join(save_root, "mean_difference_layers.pdf"),
        group1_name=group1_name,
        group2_name=group2_name,
    )

    # Best WD layer report
    best_layer = layers[int(np.argmax(wasserstein_vals))]
    best_stats = all_results[best_layer]["statistics"]
    g1b = best_stats["group1"]
    g2b = best_stats["group2"]

    md_best, md_lo, md_hi, se_md = compute_mean_diff_ci(
        g1b["mean"],
        g1b["std"],
        g1b["n"],
        g2b["mean"],
        g2b["std"],
        g2b["n"],
    )
    t_best, df_best, p_welch = compute_welch_p(
        g1b["mean"],
        g1b["std"],
        g1b["n"],
        g2b["mean"],
        g2b["std"],
        g2b["n"],
    )

    wd_val = best_stats["wass"]["D"]
    wd_p = best_stats["wass"]["p"]
    wd_ci_low, wd_ci_high = md_lo, md_hi

    d_val = best_stats["separation"]["cohen_d"]
    _, d_ci_low, d_ci_high = cohens_d_ci(d_val, g1b["n"], g2b["n"])
    d_p = p_welch

    auc_val = best_stats["auc"]
    auc_val, auc_ci_low, auc_ci_high, auc_p = auc_ci(auc_val, g1b["n"], g2b["n"])

    report = {
        "layer": int(best_layer),
        "vector_name": vector_name,
        "condition": condition_name,
        "groups": {
            "left": group1_name,
            "right": group2_name,
        },
        "WD": {
            "value": round(float(wd_val), 2),
            "pvalue": round(float(wd_p), 3),
            "95% CI": [round(float(wd_ci_low), 2), round(float(wd_ci_high), 2)],
        },
        "means": {
            group1_name: {
                "M": g1b["mean"], 
                "SD": g1b["std"], 
                "N": g1b["n"]
            },
           group2_name: {
                "M": g2b["mean"], 
                "SD": g2b["std"], 
                "N": g2b["n"]
            }
        },
        "t_test": {
        "t": round(float(t_best), 2),
        "df": round(float(df_best), 1),
        "pvalue": round(float(p_welch), 3),
        },
        "cohens_d": {
            "value": round(float(d_val), 2),
            "pvalue": round(float(d_p), 3),
            "95% CI": [round(float(d_ci_low), 2), round(float(d_ci_high), 2)],
        },
        "AUC": {
            "value": round(float(auc_val), 2),
            "pvalue": round(float(auc_p), 3),
            "95% CI": [round(float(auc_ci_low), 2), round(float(auc_ci_high), 2)],
        },
    }

    report_path = os.path.join(save_root, "report.json")
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(json.dumps(report, indent=2))


# ------------------------------------------------
# Main: iterate over conditions for a given VECTOR_NAME
# ------------------------------------------------
if __name__ == "__main__":
    # You set this manually
    VECTOR_NAME = "realistic_vs_symbolic"

    projections_root = f"./projections/treatments/{VECTOR_NAME}"
    summary_root = "./projection_results_summary/treatments"

    if not os.path.isdir(projections_root):
        raise FileNotFoundError(f"projections_root not found: {projections_root}")

    # All immediate subfolders: these are the conditions
    subfolders = [
        os.path.join(projections_root, d)
        for d in os.listdir(projections_root)
        if os.path.isdir(os.path.join(projections_root, d))
    ]

    if not subfolders:
        raise FileNotFoundError(f"No condition folders found in {projections_root}")

    for cond_path in sorted(subfolders):
        folder_name = os.path.basename(cond_path.rstrip(os.sep))

        # e.g. 'both_threat_vs_no_threat'
        condition_base = strip_timestamp(folder_name)
        condition_name = condition_base

        # left/right for labels, e.g. 'both_threat', 'no_threat'
        left_raw, right_raw = parse_condition_left_right(condition_name)

        # Output folder name, e.g. 'both_vs_no_threat'
        condition_out = condition_to_output_folder(condition_name)

        save_root = os.path.join(summary_root, VECTOR_NAME, condition_out)

        print(
            f"Processing VECTOR_NAME={VECTOR_NAME}, "
            f"condition={condition_name}, "
            f"left={left_raw}, right={right_raw} -> {save_root}"
        )

        run_one_projection(
            json_path=cond_path,
            save_root=save_root,
            group1_name=left_raw,
            group2_name=right_raw,
            vector_name=VECTOR_NAME,
            condition_name=condition_name,
        )


Processing VECTOR_NAME=realistic_vs_symbolic, condition=both_threat_vs_symbolic_only, left=both_threat, right=symbolic_only -> ./projection_results_summary/treatments/realistic_vs_symbolic/both_threat_vs_symbolic_only
{
  "layer": 31,
  "vector_name": "realistic_vs_symbolic",
  "condition": "both_threat_vs_symbolic_only",
  "groups": {
    "left": "both_threat",
    "right": "symbolic_only"
  },
  "WD": {
    "value": 0.57,
    "pvalue": 0.0,
    "95% CI": [
      -0.67,
      -0.47
    ]
  },
  "means": {
    "both_threat": {
      "M": 2.7147623697916665,
      "SD": 0.3747488788143751,
      "N": 84
    },
    "symbolic_only": {
      "M": 3.2863692434210527,
      "SD": 0.28708443418839724,
      "N": 95
    }
  },
  "t_test": {
    "t": -11.34,
    "df": 154.7,
    "pvalue": 0.0
  },
  "cohens_d": {
    "value": -1.71,
    "pvalue": 0.0,
    "95% CI": [
      -2.06,
      -1.37
    ]
  },
  "AUC": {
    "value": 0.1,
    "pvalue": 0.0,
    "95% CI": [
      0.06,
      0.15
    ]


Steering Results

In [13]:
"""
Visualize Steering Generalization Results (summary version)

Usage:
- Set JSON_PATH and OUTPUT_NAME in the __main__ block.
- Run: python visualize_steering_summary.py

Outputs (in steering_results_summary/OUTPUT_NAME/):
- violin_overall_by_alpha.png
- mean_response_vs_alpha.png
- report.json
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# ----------------------------------------------------------------------
# Global plotting style
# ----------------------------------------------------------------------
sns.set_style("white")  # no gridlines by default

plt.rcParams.update({
    # Resolution
    "figure.dpi": 300,
    "savefig.dpi": 300,

    # Font sizes (good after ~45% downscaling)
    "font.size": 12,         # base
    "axes.labelsize": 14,    # x/y labels
    "axes.titlesize": 14,    # titles (if used)
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # Line thickness
    "axes.linewidth": 1.4,   # axis spines
    "lines.linewidth": 2.4,  # main plot lines

    # Tick thickness
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,

    # Legend readability
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
})

# ----------------------------------------------------------------------
# Data loading
# ----------------------------------------------------------------------
def load_generalization_data(json_path: str) -> pd.DataFrame:
    """Load and parse generalization results from JSON into a DataFrame."""
    with open(json_path, "r") as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    # Ensure numeric types
    df["gpt4_rating"] = pd.to_numeric(df["gpt4_rating"], errors="coerce")
    df["alpha"] = pd.to_numeric(df["alpha"], errors="coerce")
    df["layer"] = pd.to_numeric(df["layer"], errors="coerce")

    # Drop invalid ratings
    df = df[df["gpt4_rating"].notna()].copy()

    print(f"✅ Loaded {len(df)} evaluations")
    print(f"   Layers: {sorted(df['layer'].dropna().unique())}")
    print(f"   Alphas: {sorted(df['alpha'].dropna().unique())}")
    print(f"   Scenarios: {len(df['scenario'].unique())}")

    return df


# ----------------------------------------------------------------------
# Plot 1: Violin plot (overall distribution by alpha)
# ----------------------------------------------------------------------
def plot_overall_violin(df: pd.DataFrame, save_path: Path):
    """
    Violin plot of GPT-4 ratings for each steering alpha,
    pooling across all layers and scenarios.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    df_plot = df.copy()
    df_plot["alpha_str"] = df_plot["alpha"].apply(lambda x: f"α={x:+.1f}")
    df_plot["gpt4_rating"] += 1

    sns.violinplot(
        data=df_plot,
        x="alpha_str",
        y="gpt4_rating",
        ax=ax,
        palette="RdYlGn_r",
        inner="box",
        cut=0,
    )

    ax.set_xlabel("Steering Strength (α)", fontsize=14, fontweight="bold")
    ax.set_ylabel("Hostility Rating", fontsize=14, fontweight="bold")
    # ax.set_title("Overall Distribution Across All Layers", fontsize=13, fontweight="bold")

    # Optional threshold line at 2.0
    ax.axhline(
        y=3.0,
        color="orange",
        linestyle="--",
        linewidth=1.5,
        # label="Hostile threshold",
        alpha=0.8,
    )
    ax.set_ylim(0.5, 5.5)
    # ax.legend(fontsize=9)
    ax.grid(False)

    # New centralized styling
    style_axes(ax)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"✅ Saved violin plot: {save_path}")


# ----------------------------------------------------------------------
# Plot 2: Mean response vs alpha (averaged across all layers)
# ----------------------------------------------------------------------

def style_axes(ax):
    """Apply minimalist, high-contrast journal style to a single axes."""
    # No gridlines
    ax.grid(False)

    # Remove top/right spines for minimalist look
    sns.despine(ax=ax, trim=True)

    # Bold tick labels for better legibility in small panels
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

    # Slightly thicker spines (in addition to rcParams)
    for spine in ax.spines.values():
        spine.set_linewidth(1.4)


def plot_mean_response_vs_alpha(df: pd.DataFrame, save_path: Path):
    """
    Mean ± 95% CI of GPT-4 rating as a function of alpha,
    pooling across all layers.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    summary = (
        df.groupby("alpha")["gpt4_rating"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("alpha")
    )

    # 95% CI using normal approximation
    summary["se"] = summary["std"] / np.sqrt(summary["count"])
    summary["ci"] = 1.96 * summary["se"]

    ax.errorbar(
        summary["alpha"],
        summary["mean"] + 1,
        yerr=summary["ci"],
        fmt="o-",
        linewidth=2.4,
        markersize=7,
        capsize=5,
        capthick=1.6,
        color="black",
        ecolor="black",
    )

    ax.set_xlabel("Steering Strength (α)", fontsize=14, fontweight="bold")
    ax.set_ylabel("Mean Hostility (±95% CI)", fontsize=14, fontweight="bold")

    # ax.set_title(
    #     "Mean GPT-4 Rating Across All Layers", fontsize=13, fontweight="bold"
    # )

    ax.axhline(
        y=3.0, color="orange", linestyle="--", linewidth=1.6, alpha=0.8, label="Hostile threshold"
    )

    # Legend: larger and easier to read
    legend = ax.legend(
        loc="upper left",
        frameon=False,   # minimalist, no box
        handlelength=2.2,
        handletextpad=0.6,
        borderpad=0.2,
    )

    for text in legend.get_texts():
        text.set_fontweight("bold")

    ax.set_ylim(0.5, 5.5)
    ax.grid(False)

    style_axes(ax)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"✅ Saved mean response plot: {save_path}")


# ----------------------------------------------------------------------
# Reporting utilities
# ----------------------------------------------------------------------
def _alpha_stats(ratings: pd.Series) -> dict:
    """Compute mean, std, n, and 95% CI for a series of ratings."""
    ratings = ratings.dropna()
    n = int(ratings.shape[0])
    if n == 0:
        return {"n": 0, "mean": None, "std": None, "ci_95": [None, None]}

    mean = float(ratings.mean())+1
    std = float(ratings.std(ddof=1))
    se = std / np.sqrt(n)
    ci_low = float(mean - 1.96 * se)
    ci_high = float(mean + 1.96 * se)
    return {"n": n, "mean": mean, "std": std, "ci_95": [ci_low, ci_high]}


def _compare_alphas(df: pd.DataFrame, alpha_high: float, alpha_low: float) -> dict:
    """Difference, Cohen's d, and p-value for alpha_high vs alpha_low."""
    x = df[df["alpha"] == alpha_high]["gpt4_rating"].dropna()
    y = df[df["alpha"] == alpha_low]["gpt4_rating"].dropna()

    if x.empty or y.empty:
        return {
            "alpha_high": alpha_high,
            "alpha_low": alpha_low,
            "mean_diff": None,
            "t_stat": None,
            "cohen_d": None,
            "p_value": None,
        }

    # Welch t-test
    t_res = stats.ttest_ind(x, y, equal_var=False)
    mean_diff = float(x.mean() - y.mean())

    # Pooled SD for Cohen's d (independent groups)
    var_x = x.var(ddof=1)
    var_y = y.var(ddof=1)
    n_x = x.shape[0]
    n_y = y.shape[0]
    pooled_var = ((n_x - 1) * var_x + (n_y - 1) * var_y) / (n_x + n_y - 2)
    if pooled_var > 0:
        d = float(mean_diff / np.sqrt(pooled_var))
    else:
        d = None

        # Welch df
    se_sq = var_x / n_x + var_y / n_y
    df_num = se_sq**2
    df_den = (var_x**2 / (n_x**2 * (n_x - 1))) + (var_y**2 / (n_y**2 * (n_y - 1)))
    df = df_num / df_den if df_den > 0 else np.nan


    return {
        "alpha_high": float(alpha_high),
        "alpha_low": float(alpha_low),
        "mean_diff": mean_diff,
        "cohen_d": d,
        "t_stat": round(float(t_res.statistic),2),
        "df": round(float(df),2),
        "p_value": float(t_res.pvalue),
    }


def _find_alpha_value(alpha_values, target, tol=1e-6):
    """Find the actual alpha in the data that matches a target value."""
    for a in alpha_values:
        if abs(a - target) < tol:
            return float(a)
    return None


def generate_alpha_report(df: pd.DataFrame, save_path: Path):
    """
    Generate report.json with:

    - Per-alpha statistics (mean, std, n, 95% CI).
    - Comparisons:
        α=+2 vs α=0
        α=+2 vs α=−2
    """
    alphas = sorted(df["alpha"].dropna().unique())
    alpha_stats_dict = {}

    # Per-alpha stats for all alphas present
    for a in alphas:
        ratings = df[df["alpha"] == a]["gpt4_rating"]
        alpha_stats_dict[str(float(a))] = _alpha_stats(ratings)

    # Focus on alpha = -2, 0, +2 (if present)
    focus_targets = [-2.0, 0.0, 2.0]
    focus_map = {t: _find_alpha_value(alphas, t) for t in focus_targets}

    comparisons = {}

    # α=+2 vs 0
    if focus_map[2.0] is not None and focus_map[0.0] is not None:
        key = f"{focus_map[2.0]}_vs_{focus_map[0.0]}"
        comparisons[key] = _compare_alphas(df, focus_map[2.0], focus_map[0.0])

    # α=+2 vs −2
    if focus_map[2.0] is not None and focus_map[-2.0] is not None:
        key = f"{focus_map[2.0]}_vs_{focus_map[-2.0]}"
        comparisons[key] = _compare_alphas(df, focus_map[2.0], focus_map[-2.0])

    report = {
        "alphas": alpha_stats_dict,
        "comparisons": comparisons,
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print(f"✅ Saved report: {save_path}")
    print(json.dumps(report, indent=2))


# ----------------------------------------------------------------------
# Main entry (manual configuration)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # ==============================================================
    # MANUAL CONFIGURATION
    # ==============================================================
    import glob
    from pathlib import Path

    # Pattern to your detailed step-3 generalization JSON file
    vector = "hostile"
    JSON_PATTERN = f"steering/{vector}/detailed_results/step3_generalization_*.json"

    # Resolve pattern -> concrete file path (take first match)
    matched_files = sorted(glob.glob(JSON_PATTERN))
    if not matched_files:
        raise FileNotFoundError(f"No JSON files match pattern: {JSON_PATTERN}")

    JSON_PATH = matched_files[0]  # take the first match
    print(f"Using JSON file: {JSON_PATH}")

    # Name for this run (used as subfolder under steering_results_summary/)
    # e.g. use file stem without extension
    OUTPUT_NAME = vector  # 'hostile'

    # Base output directory
    BASE_SUMMARY_DIR = Path("steering_results_summary")
    OUT_DIR = BASE_SUMMARY_DIR / OUTPUT_NAME
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print("STEERING GENERALIZATION SUMMARY VISUALIZATION")
    print("=" * 70)
    print(f"Input JSON pattern: {JSON_PATTERN}")
    print(f"Resolved JSON path: {JSON_PATH}")
    print(f"Output dir: {OUT_DIR}")
    print("=" * 70)

    # --------------------------------------------------------------
    # Load data
    # --------------------------------------------------------------
    df = load_generalization_data(JSON_PATH)

    # --------------------------------------------------------------
    # Plot 1: Violin plot
    # --------------------------------------------------------------
    plot_overall_violin(
        df,
        OUT_DIR / "violin_overall_by_alpha.png",
    )

    # --------------------------------------------------------------
    # Plot 2: Mean response vs alpha (averaged across all layers)
    # --------------------------------------------------------------
    plot_mean_response_vs_alpha(
        df,
        OUT_DIR / "mean_response_vs_alpha.png",
    )

    # --------------------------------------------------------------
    # Report: per-alpha stats + comparisons for α=2 vs 0 and α=2 vs −2
    # --------------------------------------------------------------
    generate_alpha_report(
        df,
        OUT_DIR / "report.json",
    )

    print("=" * 70)
    print("✅ SUMMARY VISUALIZATION COMPLETE")
    print(f"All outputs saved to: {OUT_DIR}")
    print("=" * 70)



Using JSON file: steering/hostile/detailed_results/step3_generalization_20251124_044453.json
STEERING GENERALIZATION SUMMARY VISUALIZATION
Input JSON pattern: steering/hostile/detailed_results/step3_generalization_*.json
Resolved JSON path: steering/hostile/detailed_results/step3_generalization_20251124_044453.json
Output dir: steering_results_summary/hostile
✅ Loaded 300 evaluations
   Layers: [np.int64(10), np.int64(11), np.int64(14), np.int64(15), np.int64(18)]
   Alphas: [np.float64(-2.0), np.float64(0.0), np.float64(2.0)]
   Scenarios: 20
✅ Saved violin plot: steering_results_summary/hostile/violin_overall_by_alpha.png


/tmp/ipykernel_2885/2905629336.py:96: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(


✅ Saved mean response plot: steering_results_summary/hostile/mean_response_vs_alpha.png
✅ Saved report: steering_results_summary/hostile/report.json
{
  "alphas": {
    "-2.0": {
      "n": 100,
      "mean": 1.395,
      "std": 0.4989635722822323,
      "ci_95": [
        1.2972031398326824,
        1.4927968601673176
      ]
    },
    "0.0": {
      "n": 100,
      "mean": 1.63,
      "std": 0.4852365870939097,
      "ci_95": [
        1.5348936289295936,
        1.7251063710704062
      ]
    },
    "2.0": {
      "n": 100,
      "mean": 4.4350000000000005,
      "std": 0.5845441978087879,
      "ci_95": [
        4.320429337229478,
        4.549570662770523
      ]
    }
  },
  "comparisons": {
    "2.0_vs_0.0": {
      "alpha_high": 2.0,
      "alpha_low": 0.0,
      "mean_diff": 2.805,
      "cohen_d": 5.221614955488597,
      "t_stat": 36.92,
      "df": 191.51,
      "p_value": 5.028077351963812e-89
    },
    "2.0_vs_-2.0": {
      "alpha_high": 2.0,
      "alpha_low": -2.0,
